In [1]:
# Import necessary libraries
import pandas as pd
import numpy as np
import requests
import json
import time

In [2]:
# Load in clean water level data csv that will be merged to
clean_site_data = "/capstone/aridgw/outputs/gwl_cult_1km.csv"
df_sites = pd.read_csv(clean_site_data)

In [3]:
size_km = 1

In [4]:
df_sites.groupby('region').nunique()

,year_value,site_id,depth_to_gw_ft,depth_to_gw_m,latitude,longitude,data_source,pct_cultivated
region,,,,,,,,
Arkansas Delta,21,7,144,144,7,7,1,75
Central Nebraska,21,7,146,146,7,7,1,75
SoCal_Arizona,21,8,109,109,8,8,1,38
Southern Idaho,21,4,84,84,4,4,1,48
Southern Kansas,21,10,208,208,10,10,1,114
Western Texas,21,9,187,187,9,9,1,107
Western Utah,21,5,103,103,5,5,1,63


In [5]:
# Load in 1km et data csv that will be merged to
df_annual_path = "/Users/richardmonteslemus/capstone/GW-depth-data-exploration/open_et_data/openet_data_annual_1km.csv"
df_annual = pd.read_csv(df_annual_path)

In [6]:
df_annual.drop_duplicates(inplace=True)

In [7]:
# Verify correct number of rows per year
df_annual.groupby(['year']).size().reset_index(name="n") 

,year,n
0,2000,50
1,2001,50
2,2002,50
3,2003,50
4,2004,50
5,2005,50
6,2006,50
7,2007,50
8,2008,50
9,2009,50


In [8]:
# Remove lat and long to above columns duplicates to prepare for merge 
df_annual = df_annual.drop(columns=["latitude", "longitude"])

In [9]:
df_annual

,site,year,bbox_side,open_et_version,scaled_annual_et_avg
0,KSGS.371852100505801,2000,1,2.0,781.134
1,KSGS.371852100505801,2001,1,2.0,859.635
2,KSGS.371852100505801,2002,1,2.0,787.151
3,KSGS.371852100505801,2003,1,2.0,836.748
4,KSGS.371852100505801,2004,1,2.0,666.882
...,...,...,...,...,...
1295,willcox,2021,1,2.1,611.347
1296,willcox,2022,1,2.1,799.466
1297,willcox,2023,1,2.1,861.585
1298,willcox,2024,1,2.1,825.554


In [10]:
df_sites.groupby(["site_id"]).size().reset_index(name="n") 

,site_id,n
0,KSGS.371852100505801,21
1,KSGS.372043101363101,21
2,KSGS.372539100142504,21
3,KSGS.373331098033301,21
4,KSGS.373607100565301,21
5,KSGS.374111099070401,21
6,KSGS.374125100344101,21
7,KSGS.374747100552101,21
8,KSGS.375145100485701,21
9,KSGS.375454101075401,21


In [11]:
# Merge ET data to water level data 
df_merged_et = pd.merge(
    df_sites,
    df_annual,
    left_on=['site_id', 'year_value'],
    right_on=['site', 'year'],
    how='outer'
)

In [12]:
# Remove rows containing NA in columns important for analysis
df_merged_et = df_merged_et.dropna(subset=["depth_to_gw_m", "scaled_annual_et_avg"])

In [13]:
# Change year_value to int
df_merged_et["year_value"] = df_merged_et["year_value"].astype(int)

In [14]:
# Check number of observations per site in merged df
df_merged_et.groupby(["site_id"]).size().reset_index(name="n")

,site_id,n
0,KSGS.371852100505801,21
1,KSGS.372043101363101,21
2,KSGS.372539100142504,21
3,KSGS.373331098033301,21
4,KSGS.373607100565301,21
5,KSGS.374111099070401,21
6,KSGS.374125100344101,21
7,KSGS.374747100552101,21
8,KSGS.375145100485701,21
9,KSGS.375454101075401,21


In [15]:
# Remove duplicate year column
df_merged_et = df_merged_et.drop(columns='year')
df_merged_et = df_merged_et.drop(columns='site')

In [16]:
df_merged_et

,year_value,site_id,depth_to_gw_ft,depth_to_gw_m,latitude,longitude,data_source,region,pct_cultivated,bbox_side,open_et_version,scaled_annual_et_avg
0,2000,KSGS.371852100505801,239.390000,72.966072,37.31502,-100.8505,USGS,Southern Kansas,NaN,1,2.0,781.134
1,2001,KSGS.371852100505801,241.960000,73.749408,37.31502,-100.8505,USGS,Southern Kansas,NaN,1,2.0,859.635
2,2002,KSGS.371852100505801,242.780000,73.999344,37.31502,-100.8505,USGS,Southern Kansas,NaN,1,2.0,787.151
3,2003,KSGS.371852100505801,246.710000,75.197208,37.31502,-100.8505,USGS,Southern Kansas,NaN,1,2.0,836.748
4,2004,KSGS.371852100505801,247.710000,75.502008,37.31502,-100.8505,USGS,Southern Kansas,NaN,1,2.0,666.882
...,...,...,...,...,...,...,...,...,...,...,...,...
1288,2014,willcox,346.899946,105.735100,32.03619,-109.7540,Jasechko,SoCal_Arizona,63.636364,1,2.0,942.466
1289,2015,willcox,352.000011,107.289600,32.03619,-109.7540,Jasechko,SoCal_Arizona,16.804408,1,2.0,735.658
1291,2017,willcox,372.750012,113.614200,32.03619,-109.7540,Jasechko,SoCal_Arizona,22.497704,1,2.0,530.228
1293,2019,willcox,392.100078,119.512100,32.03619,-109.7540,Jasechko,SoCal_Arizona,19.559229,1,2.0,591.662


In [17]:
# Turn merged df into csv
# Convert monthly ET data into csv
df_merged_et.to_csv(f"/Users/richardmonteslemus/capstone/GW-depth-data-exploration/open_et_data/merged_openet_data_{size_km}km.csv", index=False)
# df_merged_et.to_csv(f"GW-depth-data-exploration/open_et_data/merged_openet_data_{size_km}km.csv", index=False)

